In [56]:
import pandas as pd
import numpy as np

In [57]:
#import data from /Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/ProcessedObservedData.csv
data = pd.read_csv(r'C:\Users\jaskew\Documents\project_repository\data\processed\ProcessedObservedData.csv')
#filter by opDiv CDC
data = data[data['OpDiv'] == 'CDC']
data.drop(columns=['curr_date', 'OpDiv'], inplace=True)
# Rename the index to 'date' since 'obs_date' is now the index
data.rename(columns={'obs_date': 'date'}, inplace=True)
data['date'] = pd.to_datetime(data['date'])
data.reset_index(drop=True, inplace=True)
data.head(10)
# Create a copy of the relevant columns to avoid SettingWithCopyWarning

data.head(10)

,indicator,API_UserName,date,observations
0,185.230.63.171,00818860012482918321,2024-01-01,2
1,104.21.57.152,00818860012482918321,2024-01-02,10
2,185.230.63.171,00818860012482918321,2024-01-02,18
3,192.229.221.95,00818860012482918321,2024-01-02,3
4,216.24.57.253,00818860012482918321,2024-01-02,43
5,216.24.57.3,00818860012482918321,2024-01-02,99
6,67.225.140.4,00818860012482918321,2024-01-02,3
7,104.18.32.68,00818860012482918321,2024-01-03,12
8,104.26.5.196,00818860012482918321,2024-01-03,1
9,185.230.63.171,00818860012482918321,2024-01-03,10


In [58]:
data['date'] = pd.to_datetime(data['date'])

# Define ranges for combinations
all_users = data['API_UserName'].unique()  # Unique API_UserName
all_indicators = data['indicator'].unique()  # Unique indicators
all_dates = pd.date_range(start='2025-01-01', end=pd.Timestamp.now().normalize(), freq='D')

# Create all combinations
all_combinations = pd.MultiIndex.from_product(
    [all_users, all_dates, all_indicators],
    names=['API_UserName', 'date', 'indicator']
).to_frame(index=False)

# Merge with the existing data
merged = all_combinations.merge(data, how='left', on=['API_UserName', 'date', 'indicator'])

# Fill missing values in 'observations' with 0 (not seen)
merged['observations'] = merged['observations'].fillna(0).astype(int)

# Display the first few rows of the merged dataset

# Convert the 'date' column to datetime format
merged['date'] = pd.to_datetime(merged['date'])

# Extract day of the week (0=Monday, 6=Sunday)
merged['dayofweek'] = merged['date'].dt.dayofweek

# Determine if the day is a weekend (Saturday=5, Sunday=6)
merged['is_weekend'] = merged['dayofweek'].isin([5, 6])

# Extract additional features if needed
merged['day'] = merged['date'].dt.day
merged['month'] = merged['date'].dt.month

merged['seen'] = (merged['observations'] > 0).astype(int)
merged.head(10)
merged.to_csv(r'C:\Users\jaskew\Documents\project_repository\notebooks\observationEventForecasting\DataPreprocessing\FullIndicatorMatrix.csv', index=False)